<a href="https://colab.research.google.com/github/Haross/sql_notebook/blob/main/SQL_challenge_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL In-Class Challenge 2 – All Topics

Today you will work as **SQL analysts** for a language-learning startup.

Ruby is the creator of a **language tandem app** — a platform where people learning languages can connect with native speakers and practice together through chat.

Ruby now wants to better understand how the app is being used so she can improve the platform and grow the business.

She has asked you to analyze the app’s database and answer important questions such as:

- How many countries have Spanish speakers?
- How many chatrooms are active?
- Which speakers could be good language-exchange partners?

The answers are hidden in the database — and **your SQL skills will uncover them**.

---
## Challenge format

There are **11 SQL challenges** available.

- You may solve the exercises **in any order**.
- Each correct solution gives **points for the leaderboard** depending on its difficulty.
- The **final leaderboard will be shown at the end of class**.
- 🏆 The top student will receive a **small bonus**.

⚠️ **Exercise 11 is also the graded exercise for this activity**, so make sure you complete it.

Your goal: **solve as many challenges as you can and climb the leaderboard.**


## **SQL Environment Setup (do not edit)**

In [53]:
# @title

%%capture
!mkdir -p notebook_lib
!wget --cache=off -q -O notebook_lib/sql_runner.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner.py
!wget --cache=off -q -O notebook_lib/sql_runner_store.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner_store.py
!wget --cache=off -q -O notebook_lib/sql_runner_ui_bits.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner_ui_bits.py
!wget --cache=off -q -O notebook_lib/validators.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/validators.py

!wget --cache=off -q -O notebook_lib/cloud_submitter.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/cloud_submitter.py

!touch notebook_lib/__init__.py
#!pip install -q duckdb

from notebook_lib.sql_runner import make_sql_runner
from notebook_lib.validators import make_df_validator_nospoilers, check_process_rules
from notebook_lib.cloud_submitter import make_cloud_run_submitter
import duckdb
import pandas as pd
from pathlib import Path


In [54]:
# @title

DB_FILE = 'class_joins.duckdb'

if DB_FILE != ":memory:":
    Path(DB_FILE).unlink(missing_ok=True)

conn = duckdb.connect(DB_FILE)

conn.execute(r'''
-- DuckDB schema + seed data

DROP TABLE IF EXISTS chatroom_speaker;
DROP TABLE IF EXISTS language_speaker;
DROP TABLE IF EXISTS chatroom;
DROP TABLE IF EXISTS speaker;
DROP TABLE IF EXISTS language;

CREATE TABLE speaker (
    id INTEGER PRIMARY KEY,
    name VARCHAR NOT NULL,
    email VARCHAR NOT NULL,
    country VARCHAR NOT NULL,
    city VARCHAR NOT NULL
);

CREATE TABLE language (
    id INTEGER PRIMARY KEY,
    name VARCHAR NOT NULL
);

CREATE TABLE chatroom (
    id INTEGER PRIMARY KEY,
    created_at TIMESTAMP NOT NULL,
    last_active_at TIMESTAMP NOT NULL
);

CREATE TABLE language_speaker (
    speaker_id INTEGER NOT NULL,
    language_id INTEGER NOT NULL,
    native BOOLEAN NOT NULL,
    PRIMARY KEY (speaker_id, language_id),
    FOREIGN KEY (speaker_id) REFERENCES speaker(id),
    FOREIGN KEY (language_id) REFERENCES language(id)
);

CREATE TABLE chatroom_speaker (
    speaker_id INTEGER NOT NULL,
    chatroom_id INTEGER NOT NULL,
    PRIMARY KEY (speaker_id, chatroom_id),
    FOREIGN KEY (speaker_id) REFERENCES speaker(id),
    FOREIGN KEY (chatroom_id) REFERENCES chatroom(id)
);

INSERT INTO speaker (id, name, email, country, city) VALUES
    (1, 'Karli Roberson', 'karli.roberson@example.com', 'USA', 'New York'),
    (2, 'Emma Dumont', 'emma.dumont@example.com', 'USA', 'New York'),
    (3, 'Madeleine Renaud', 'm.renaud@example.com', 'USA', 'New York'),
    (4, 'Anna Kowalska', 'a.kowalska@example.com', 'Poland', 'Warsaw'),
    (5, 'Alba Rodriguez', 'a.rodriguez@example.com', 'Spain', 'Barcelona'),
    (6, 'Gabriel Navarra', 'gabriel.nav@example.com', 'Spain', 'Barcelona'),
    (7, 'Beatrice Fabri', 'b.fabri@example.com', 'Poland', 'Warsaw'),
    (8, 'Lina Pereira', 'pereira@example.com', 'Poland', 'Warsaw'),
    (9, 'Giovanni Ferretti', 'feretti@example.com', 'Spain', 'Madrid'),
    (10, 'Maya Martinazzo', 'maya.m@example.com', 'Italy', 'Naples'),
    (11, 'Erika Weber', 'eweber@example.com', 'Italy', 'Naples'),
    (12, 'Constance Deparrois', 'c.depa@example.com', 'France', 'Paris'),
    (13, 'Julia Nowak', 'rverano@example.com', 'France', 'Paris'),
    (14, 'Susan Gray', 'susan.gray@example.com', 'France', 'Paris'),
    (15, 'Patrick Turner', 'pat.turner@example.com', 'France', 'Paris');

INSERT INTO language (id, name) VALUES
    (1, 'English'),
    (2, 'Polish'),
    (3, 'Spanish'),
    (4, 'French'),
    (5, 'German'),
    (6, 'Italian');

INSERT INTO chatroom (id, created_at, last_active_at) VALUES
    (1, '2015-01-10 00:55:58', '2020-05-25 07:46:00'),
    (2, '2020-02-11 09:07:12', '2022-06-01 16:55:29'),
    (3, '2020-02-10 21:12:52', '2020-08-27 19:56:03'),
    (4, '2019-03-31 18:28:18', '2022-02-11 14:40:15'),
    (5, '2017-04-18 21:58:36', '2021-06-17 23:49:00'),
    (6, '2018-03-01 09:00:34', '2018-11-18 18:45:04'),
    (7, '2019-10-14 17:35:12', '2020-09-19 16:36:59'),
    (8, '2020-06-13 12:32:56', '2021-01-07 14:15:02'),
    (9, '2020-01-20 00:05:19', '2020-12-22 01:19:16'),
    (10, '2022-01-03 02:00:04', '2022-03-01 11:59:00');

INSERT INTO language_speaker (speaker_id, language_id, native) VALUES
    (1, 1, TRUE),
    (2, 1, FALSE),
    (2, 5, FALSE),
    (2, 4, TRUE),
    (3, 4, TRUE),
    (3, 1, TRUE),
    (3, 5, FALSE),
    (4, 2, TRUE),
    (4, 1, FALSE),
    (4, 4, FALSE),
    (4, 5, FALSE),
    (5, 3, TRUE),
    (5, 1, FALSE),
    (5, 6, FALSE),
    (6, 3, TRUE),
    (6, 1, FALSE),
    (7, 6, TRUE),
    (7, 2, TRUE),
    (7, 1, FALSE),
    (7, 4, FALSE),
    (9, 6, TRUE),
    (9, 3, FALSE),
    (10, 6, TRUE),
    (10, 1, FALSE),
    (11, 5, TRUE),
    (11, 6, FALSE),
    (12, 4, TRUE),
    (13, 2, TRUE),
    (13, 4, TRUE),
    (13, 1, FALSE),
    (14, 1, TRUE),
    (14, 4, FALSE),
    (15, 1, TRUE),
    (15, 4, TRUE),
    (15, 3, FALSE);

INSERT INTO chatroom_speaker (speaker_id, chatroom_id) VALUES
    (5, 1),
    (9, 1),
    (5, 2),
    (15, 2),
    (4, 3),
    (15, 3),
    (7, 4),
    (15, 4),
    (2, 5),
    (14, 5),
    (6, 6),
    (15, 6),
    (13, 7),
    (14, 7),
    (3, 8),
    (4, 8),
    (3, 9),
    (7, 9),
    (2, 10),
    (7, 10);
''')
print(f"Duckdb database ready ✅ ({DB_FILE})")


Duckdb database ready ✅ (class_joins.duckdb)


In [55]:
# @title Cloud submission config (do not edit)
CLOUD_SUBMIT_URL = 'https://cinemax-validator-830800716261.europe-west1.run.app/submit'
CLOUD_EXAM_ID = 'SQL_CHALLENGE_2_2026'
CLOUD_API_KEY = 'SECRET123'


In [56]:
# @title
import ipywidgets as widgets
from IPython.display import display, HTML
from pathlib import Path

TOKEN_FILE = Path("student_token.txt")

status_out = widgets.Output()

def update_status(title, message, type="info"):
    icons = {
        "info": "ℹ️",
        "success": "✅",
        "warning": "⚠️",
        "error": "❌",
    }

    icon = icons.get(type, "ℹ️")

    with status_out:
        status_out.clear_output()
        display(HTML(f"""
        <style>
        /* ===== Theme Variables ===== */
        html {{
            --info-border: #1a73e8;
            --info-bg: #eef3fe;

            --success-border: #2e7d32;
            --success-bg: #eaf7ee;

            --warning-border: #f9a825;
            --warning-bg: #fff8e1;

            --error-border: #c62828;
            --error-bg: #fdecea;

            --text-color: #202124;
            --shadow: 0 2px 6px rgba(0,0,0,0.05);
        }}

        html[theme="dark"] {{
            --info-border: #8ab4f8;
            --info-bg: #1e2a3a;

            --success-border: #81c995;
            --success-bg: #1f3324;

            --warning-border: #fdd663;
            --warning-bg: #3a3218;

            --error-border: #f28b82;
            --error-bg: #3a1f1f;

            --text-color: #e8eaed;
            --shadow: 0 2px 8px rgba(0,0,0,0.6);
        }}

        /* ===== Status Box ===== */
        .status-box {{
            padding:16px;
            border-left:6px solid var(--{type}-border);
            background: var(--{type}-bg);
            border-radius:10px;
            font-family:Arial, sans-serif;
            line-height:1.6;
            box-shadow: var(--shadow);
            color: var(--text-color);
        }}

        .status-title {{
            font-size:16px;
            font-weight:600;
            margin-bottom:6px;
        }}

        .status-message {{
            font-size:14px;
        }}
        </style>

        <div class="status-box">
            <div class="status-title">
                {icon} {title}
            </div>
            <div class="status-message">
                {message}
            </div>
        </div>
        """))

display(status_out)


t = TOKEN_FILE.read_text().strip() if TOKEN_FILE.exists() else ""

update_status(
    "Important: Before Starting",
    """
    The next cell will ask you to enter your <b>student token</b>.<br><br>
    After typing it, press <b>Enter</b>.<br>
    You only need to do this once.
    """,
    type="info"
)

Output()

In [57]:
# @title
from pathlib import Path

TOKEN_FILE = Path("student_token.txt")

existing = TOKEN_FILE.read_text().strip() if TOKEN_FILE.exists() else ""
if not existing:
    while True:
        token = input("Student token: ").strip()
        if token:
            break
        print("⚠️ Token cannot be empty. Try again.")

    TOKEN_FILE.write_text(token)
    update_status(
        "Token Saved Successfully",
        f"Your token has been stored securely.<br><br>Your token is:<b>{token}</b>",
        type="success"
    )
    print("✅ Token saved.")
else:
    token = existing
    update_status(
        "Token Already Saved",
        f"Your token has been already stored securely.<br><br>Your token is: <b>{token}</b>",
        type="success"
    )
    print("✅ Token already saved.")

✅ Token already saved.


## Get to know the tables

### The database and the `speaker` table

The app database consists of five tables:

- `speaker`
- `language`
- `language_speaker`
- `chatroom`
- `chatroom_speaker`

Let’s take a closer look at each of them.

### `speaker`

The first table, `speaker`, contains information about the users of the app.

It has the following columns:

- `id` – the unique identifier of each speaker  
- `name` – the speaker’s name  
- `email` – the speaker’s email address  
- `country` – the country where the speaker is located  
- `city` – the city where the speaker is located  

---

### The `language` table

The table `language` stores the languages available in the app.

It contains the following columns:

- `id` – the unique identifier of each language  
- `name` – the name of the language  

---

### The `language_speaker` table

The table `language_speaker` connects speakers with languages.

It contains the following columns:

- `speaker_id` – the id of the speaker  
- `language_id` – the id of the language  
- `native` – a Boolean value indicating whether the speaker is a **native speaker** of that language or a **learner**

This table allows one speaker to be linked to multiple languages.

---

### The `chatroom` table

The table `chatroom` contains information about the conversations.

It has the following columns:

- `id` – the unique identifier of the chatroom  
- `created_at` – the timestamp when the chatroom was created  
- `last_active_at` – the timestamp of the most recent message sent in the chatroom  

Note that each chatroom can contain **exactly two speakers**. Group chats are not allowed.

---

### The `chatroom_speaker` table

Finally, the table `chatroom_speaker` connects speakers with chatrooms.

It contains the following columns:

- `speaker_id` – the id of the speaker  
- `chatroom_id` – the id of the chatroom  

This table tells us which two speakers participate in each chatroom.


In [58]:
# @title ER diagram
%%html
<img id="er-img" style="width:50%; max-width:100%;  height:auto;"
     data-light="https://raw.githubusercontent.com/Haross/DB_pics_nt/main/ER_sql_challenge_1.png"
     data-dark="https://raw.githubusercontent.com/Haross/DB_pics_nt/main/ER_sql_challenge_1_black.png"
     alt="ER diagram">

<script>
  const img = document.getElementById("er-img");

  function isDarkTheme() {
    // Colab sets html[theme=dark] on the top document
    const themeAttr = document.documentElement.getAttribute("theme");
    if (themeAttr) return themeAttr === "dark";

    // fallback: OS/browser preference
    return window.matchMedia && window.matchMedia("(prefers-color-scheme: dark)").matches;
  }

  function updateImage() {
    img.src = isDarkTheme() ? img.dataset.dark : img.dataset.light;
  }

  updateImage();

  // React to Colab theme toggles (attribute changes)
  new MutationObserver(updateImage).observe(document.documentElement, {
    attributes: true,
    attributeFilter: ["theme"]
  });

  // React to OS/browser theme changes (fallback)
  if (window.matchMedia) {
    const mq = window.matchMedia("(prefers-color-scheme: dark)");
    mq.addEventListener?.("change", updateImage);
    mq.addListener?.(updateImage); // older browsers
  }
</script>

In [59]:
# @title In-class exercise 2 (10 points)
submitter_in_class_ex_2 = make_cloud_run_submitter(
    submit_url=CLOUD_SUBMIT_URL,
    exam_id=CLOUD_EXAM_ID,
    question_id='ex2',
    api_key=CLOUD_API_KEY,
)

make_sql_runner(
    conn,
    runner_id="in_class_ex_2",
    description_md='### In-class exercise 2 (10 points - Medium)\nAs the next step in her promotional campaign, Ruby has asked you to create a report showing the languages along with the number of countries in which you can practice each respective language with native speakers. She is only interested in those with native speakers in more than one country.\n\nDisplay the name of the language and the number of different countries in which it can be practiced with native speakers (name the column country_count). Sort the results by the number of countries in ascdending order. Languages with native speakers in only one country (or fewer) should not be included in this report.\n',
    submitter=submitter_in_class_ex_2,
    sol_sql=None,
    select_only=True,
    dedupe=True
)


In [60]:
# @title In-class exercise 3 (5 points) - Medium
submitter_in_class_ex_3 = make_cloud_run_submitter(
    submit_url=CLOUD_SUBMIT_URL,
    exam_id=CLOUD_EXAM_ID,
    question_id='ex3',
    api_key=CLOUD_API_KEY,
)

make_sql_runner(
    conn,
    runner_id="in_class_ex_3",
    description_md='### In-class exercise 3 (5 points - Medium)\nThis time, Ruby would like to find out how many speakers may communicate in English in their chatrooms. She has asked you to find the number of chatrooms in which both participants are English speakers (native or not). Are you ready?\n\nDisplay the number of chatrooms in which both participants are English speakers (native or not). Call this column english_speaking_chatrooms_count.\n',
    submitter=submitter_in_class_ex_3,
    sol_sql=None,
    select_only=True,
    dedupe=True
)


In [61]:
# @title In-class exercise 5 (10 points)
submitter_in_class_ex_5 = make_cloud_run_submitter(
    submit_url=CLOUD_SUBMIT_URL,
    exam_id=CLOUD_EXAM_ID,
    question_id='ex5',
    api_key=CLOUD_API_KEY,
)

make_sql_runner(
    conn,
    runner_id="in_class_ex_5",
    description_md="### In-class exercise 5 (10 points - Medium)\nRuby is interested in the number of active chatrooms in which each user participates. A chatroom is considered active if at least one message has been sent in 2020 and later. Let's find out!\n\nDisplay the speaker's name and the count of chatrooms in which the speaker participates. Only count the chatrooms that have been active, defined by the last message sent in 2020 or later. Call this column chatroom_count. Sort the results by chatroom_count in descending order.\n",
    submitter=submitter_in_class_ex_5,
    sol_sql=None,
    select_only=True,
    dedupe=True
)


In [62]:
# @title In-class exercise 6 (10 points)
submitter_in_class_ex_6 = make_cloud_run_submitter(
    submit_url=CLOUD_SUBMIT_URL,
    exam_id=CLOUD_EXAM_ID,
    question_id='ex6',
    api_key=CLOUD_API_KEY,
)

make_sql_runner(
    conn,
    runner_id="in_class_ex_6",
    description_md="### In-class exercise 6 (10 points - Medium) \nNow, let's modify the previous task a bit. Can you find the average number of active chatrooms per user? As a reminder, a chatroom is considered active if at least one message has been sent in 2020 and later.\n\nDisplay the average number of active chatrooms per user. A chatroom is considered active if at least one message has been sent in 2020 and later. The column displayed should be called avg_chatroom_count. Round the result to 2 decimal places. Note there may be users who do not participate in any chats (or any active chats) at all; we would like to display 0 for those users instead of omitting them.\n",
    submitter=submitter_in_class_ex_6,
    sol_sql=None,
    select_only=True,
    dedupe=True
)


In [63]:
# @title In-class exercise 9 (5 points)
submitter_in_class_ex_9 = make_cloud_run_submitter(
    submit_url=CLOUD_SUBMIT_URL,
    exam_id=CLOUD_EXAM_ID,
    question_id='ex9',
    api_key=CLOUD_API_KEY,
)

make_sql_runner(
    conn,
    runner_id="in_class_ex_9",
    description_md='### In-class exercise 9 (5 points - Easy)\nRuby is experimenting with recommendation filters for the app. She would like to identify speakers that match certain geographic and naming patterns. Display the speaker\'s **name**, **country**, and **city**.\n\nReturn speakers that satisfy the following criteria:\n\n* The speaker lives in **USA or Spain**\n* The speaker lives in **New York or Barcelona**\n* The speaker\'s **name starts with "A" or "M"**\n\nAdditionally, include speakers who:\n\n* live in **France**\n* live in **Paris**\n* have an **email containing "example.com"**\n',
    submitter=submitter_in_class_ex_9,
    sol_sql=None,
    select_only=True,
    dedupe=True
)


In [64]:
# @title In-class exercise 11 (15 points)
submitter_in_class_ex_11 = make_cloud_run_submitter(
    submit_url=CLOUD_SUBMIT_URL,
    exam_id=CLOUD_EXAM_ID,
    question_id='ex11',
    api_key=CLOUD_API_KEY,
)

make_sql_runner(
    conn,
    runner_id="in_class_ex_11",
    description_md="### In-class exercise 11 (15 points - Hard) \nRuby is preparing recommendations for language exchange partners. She would like to identify speakers based on combinations of language skills and location.\n\nDisplay the **speaker's name**.\n\nReturn speakers that match the following rules:\n\n* The speaker **speaks English (language_id = 1)**\n* The speaker is **either a native speaker of English or also speaks French (language_id = 4)**\n\nAlso include speakers who:\n\n* **speak Spanish (language_id = 3)**\n* **are not native Spanish speakers**\n* live in **Spain or France**\n\nSort the results by **name**.\n",
    submitter=submitter_in_class_ex_11,
    sol_sql=None,
    select_only=True,
    dedupe=True
)


In [65]:
# @title In-class exercise 12 (15 points)
submitter_in_class_ex_12 = make_cloud_run_submitter(
    submit_url=CLOUD_SUBMIT_URL,
    exam_id=CLOUD_EXAM_ID,
    question_id='ex12',
    api_key=CLOUD_API_KEY,
)

make_sql_runner(
    conn,
    runner_id="in_class_ex_12",
    description_md='### In-class exercise 12 (15 points - Hard)\nRuby is now looking for the best **language-exchange chatrooms**. She only wants chatrooms where the two participants are from **different countries**, can already communicate in at least one **shared language**, but do **not** share any language that is **native for both of them**.\n\nDisplay the **chatroom id** and the number of shared languages between its two participants. Call this column **shared_language_count**.\n\nA chatroom should be included only if:\n\n* it has exactly **two participants**\n* the two participants are from **different countries**\n* they share **at least one language**\n* there is **no language** that is marked as **native for both participants**\n\nSort the results by **shared_language_count** in descending order, and then by **id** in ascending order.\n\n💡 Hint: This problem combines ideas from earlier exercises in SQL Challenge 1. Reviewing how speaker pairs were created (exercise 8) and how shared languages were analyzed (exercise 7) may help.\n',
    submitter=submitter_in_class_ex_12,
    sol_sql=None,
    select_only=True,
    dedupe=True
)
